# RAM Graph Energy — E0/E1 (puerta bloqueante)

Este notebook **no entrena**. Comprueba que las fórmulas que se van a optimizar, proyectar y medir coinciden con la especificación. No continúes al piloto si este notebook falla.


## 1. Instalar el paquete


In [ ]:
import os, pathlib, shutil, subprocess, sys

REPO_URL = "https://github.com/abelalv/energia-2.git"
REPO_DIR = pathlib.Path("/kaggle/working/energia-2")

if not (REPO_DIR / "pyproject.toml").exists():
    candidates = [
        path
        for path in pathlib.Path("/kaggle/input").rglob("pyproject.toml")
        if (path.parent / "src" / "ram_experiments").exists()
    ]
    if candidates:
        shutil.copytree(candidates[0].parent, REPO_DIR, dirs_exist_ok=True)
        print("repo copiado desde", candidates[0].parent)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
# Los subprocesos del notebook deben encontrar el paquete también sin instalación editable.
os.environ["PYTHONPATH"] = os.pathsep.join(
    [str(REPO_DIR / "src"), os.environ.get("PYTHONPATH", "")]
)
try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
        check=True,
    )
except subprocess.CalledProcessError:
    # Kaggle sin red o sin uv_build: usar src/ directamente. El entorno sigue siendo el de Kaggle.
    sys.path.insert(0, str(REPO_DIR / "src"))
    print("pip install -e falló; usando src/ en sys.path")
print("cwd:", pathlib.Path.cwd())


## 2. Entorno y GPU


In [ ]:
import platform, torch

print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
    print("vram GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

# Protocol RAM-2026-10-13-v1: float32 without AMP/TF32.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print("tf32:", torch.backends.cuda.matmul.allow_tf32)


## 3. Verificaciones E0/E1 y checks matemáticos preservados


In [ ]:
import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "ram_experiments.cli",
        "verify",
        "--suite",
        "E0,E1",
        "--out",
        "/kaggle/working/E0",
    ],
    check=True,
)
subprocess.run([sys.executable, "mathematical_checks/verify_mathematics.py"], check=True)


## 4. Lectura del informe

`/kaggle/working/E0/E0_E1_report.json` debe tener `status = passed`. Los 227 checks de `mathematical_checks` complementan, no sustituyen, E0/E1.


In [ ]:
import json, pathlib
path = pathlib.Path('/kaggle/working/E0/E0_E1_report.json')
report = json.loads(path.read_text())
print(report['status'], report['returncode'])
